# 04 - Compute Graph Analytics

This notebook computes reusable graph analytics for the application after PrimeKG has been loaded into Neo4j.

Outputs written back to Neo4j:

- `degree`: raw undirected relationship count for each `:Entity` node.
- `degree_centrality`: normalized degree score using the largest observed degree.
- `degree_size`: capped log-scaled display size for Cytoscape nodes.
- `SIMILAR_DISEASE`: precomputed disease-to-disease similarity relationships with explainable shared evidence counts.

## 1. Connection Configuration

Use the same Neo4j environment variables as the loading notebook.

In [1]:
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

GDS_SIMILARITY_GRAPH = "biokg_disease_similarity"
SIMILARITY_TOP_K = 15
SIMILARITY_CUTOFF = 0.05
EVIDENCE_NODE_TYPES = ["gene/protein", "pathway", "effect/phenotype"]

print(f"Neo4j URI: {NEO4J_URI}")
print(f"Neo4j database: {NEO4J_DATABASE}")

Neo4j URI: neo4j://127.0.0.1:7687
Neo4j database: neo4j


## 2. Connect To Neo4j

Create the driver and verify that Neo4j is reachable before writing analytics properties.

In [2]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j")

Connected to Neo4j


## 3. Compute Degree-Based Display Sizes

The raw degree can be very skewed in biomedical graphs, so the display size uses a capped logarithmic transform. This preserves hub importance without letting large hubs dominate the canvas.

In [3]:
degree_query = """
MATCH (node:Entity)
WITH node, size([(node)--() | 1]) AS degree
WITH collect({node: node, degree: degree}) AS rows, max(degree) AS max_degree
UNWIND rows AS row
WITH row.node AS node, row.degree AS degree, max_degree
SET
    node.degree = degree,
    node.degree_centrality = CASE WHEN max_degree = 0 THEN 0.0 ELSE toFloat(degree) / max_degree END,
    node.degree_size = 28.0 + 42.0 * (log10(1.0 + degree) / log10(1.0 + max_degree))
RETURN count(node) AS updated_nodes, max_degree
"""

with driver.session(database=NEO4J_DATABASE) as session:
    record = session.run(degree_query).single()

print(dict(record))

{'updated_nodes': 129375, 'max_degree': 17355}


## 4. Reset Existing Similarity Relationships

Delete previous computed disease similarity relationships so the notebook can be rerun idempotently.

In [4]:
with driver.session(database=NEO4J_DATABASE) as session:
    deleted = session.run("""
    MATCH ()-[rel:SIMILAR_DISEASE]-()
    WITH rel LIMIT 50000
    DELETE rel
    RETURN count(rel) AS deleted_relationships
    """).single()

print(dict(deleted))

{'deleted_relationships': 0}


## 5. Project Disease Evidence Graph For GDS

The projection keeps disease nodes plus interpretable evidence nodes. Disease similarity will be based on shared genes/proteins, pathways, and phenotypes.

In [5]:
projection_query = """
CALL gds.graph.project.cypher(
    $graph_name,
    'MATCH (node:Entity)
     WHERE node.node_type = "disease" OR node.node_type IN $evidence_node_types
     RETURN id(node) AS id',
    'MATCH (source:Entity)-[rel]-(target:Entity)
     WHERE source.node_type = "disease" AND target.node_type IN $evidence_node_types
     RETURN id(source) AS source, id(target) AS target
     UNION
     MATCH (source:Entity)-[rel]-(target:Entity)
     WHERE target.node_type = "disease" AND source.node_type IN $evidence_node_types
     RETURN id(target) AS source, id(source) AS target',
    {parameters: {evidence_node_types: $evidence_node_types}}
)
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
"""

with driver.session(database=NEO4J_DATABASE) as session:
    exists = session.run(
        "CALL gds.graph.exists($graph_name) YIELD exists RETURN exists",
        graph_name=GDS_SIMILARITY_GRAPH,
    ).single()["exists"]
    if exists:
        session.run("CALL gds.graph.drop($graph_name) YIELD graphName RETURN graphName", graph_name=GDS_SIMILARITY_GRAPH).consume()
    projection = session.run(
        projection_query,
        graph_name=GDS_SIMILARITY_GRAPH,
        evidence_node_types=EVIDENCE_NODE_TYPES,
    ).single()

print(dict(projection))

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. gds.graph.project.cypher is deprecated. It is replaced by gds.graph.project Cypher projection as an aggregation function.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nCALL gds.graph.project.cypher(\n    $graph_name,\n    \'MATCH (node:Entity)\n     WHERE node.node_type = "disease" OR node.node_type IN $evidence_node_types\n     RETURN id(node) AS id\',\n    \'MATCH (source:Entity)-[rel]-(target:Entity)\n     WHERE source.node_type = "disease" AND tar

{'graphName': 'biokg_disease_similarity', 'nodeCount': 62578, 'relationshipCount': 231749}


## 6. Stream And Store Disease Similarity

`gds.nodeSimilarity.stream` computes Jaccard-like shared-neighborhood similarity. We filter to disease-disease pairs and write bounded `SIMILAR_DISEASE` relationships for fast application queries.

In [6]:
similarity_query = """
CALL gds.nodeSimilarity.stream($graph_name, {
    topK: $top_k,
    similarityCutoff: $similarity_cutoff
})
YIELD node1, node2, similarity
WITH gds.util.asNode(node1) AS disease_a, gds.util.asNode(node2) AS disease_b, similarity
WHERE disease_a.node_type = 'disease'
  AND disease_b.node_type = 'disease'
  AND disease_a.primekg_index < disease_b.primekg_index
MERGE (disease_a)-[rel:SIMILAR_DISEASE]-(disease_b)
SET
    rel.score = similarity,
    rel.method = 'gds.nodeSimilarity',
    rel.evidence_node_types = $evidence_node_types,
    rel.updated_at = datetime()
RETURN count(rel) AS similarity_relationships
"""

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(
        similarity_query,
        graph_name=GDS_SIMILARITY_GRAPH,
        top_k=SIMILARITY_TOP_K,
        similarity_cutoff=SIMILARITY_CUTOFF,
        evidence_node_types=EVIDENCE_NODE_TYPES,
    ).single()

print(dict(result))

{'similarity_relationships': 55701}


## 7. Annotate Similarity Relationships With Evidence Counts

Add interpretable shared-evidence counts that the backend can show in the similarity panel.

In [7]:
evidence_counts_query = """
MATCH (disease_a:Entity)-[rel:SIMILAR_DISEASE]-(disease_b:Entity)
WHERE id(disease_a) < id(disease_b)
OPTIONAL MATCH (disease_a)--(gene:Entity {node_type: 'gene/protein'})--(disease_b)
WITH disease_a, disease_b, rel, count(DISTINCT gene) AS shared_gene_count
OPTIONAL MATCH (disease_a)--(pathway:Entity {node_type: 'pathway'})--(disease_b)
WITH disease_a, disease_b, rel, shared_gene_count, count(DISTINCT pathway) AS shared_pathway_count
OPTIONAL MATCH (disease_a)--(phenotype:Entity {node_type: 'effect/phenotype'})--(disease_b)
WITH rel, shared_gene_count, shared_pathway_count, count(DISTINCT phenotype) AS shared_phenotype_count
SET
    rel.shared_gene_count = shared_gene_count,
    rel.shared_pathway_count = shared_pathway_count,
    rel.shared_phenotype_count = shared_phenotype_count,
    rel.evidence_count = shared_gene_count + shared_pathway_count + shared_phenotype_count
RETURN count(rel) AS annotated_relationships
"""

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(evidence_counts_query).single()

print(dict(result))

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=3, column=7, offset=73>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 73, 'line': 3, 'column': 7}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (disease_a:Entity)-[rel:SIMILAR_DISEASE]-(disease_b:Entity)\nWHERE id(disease_a) < id(disease_b)\nOPTIONAL MATCH (disease_a)--(gene:Entity {node_type: 'gene/protein'})--(disease_b)\nWITH disease_a, disease_b, rel, count(DISTINCT gene) AS shared_gene_count\nOPTIONAL MATCH (disease_a)--(pathway:Entity {node_t

{'annotated_relationships': 55701}


## 8. Validate Analytics Outputs

Confirm that degree properties and disease similarity relationships are available for the application.

In [8]:
validation_query = """
MATCH (node:Entity)
WITH
    count(node) AS total_nodes,
    count(node.degree) AS nodes_with_degree,
    count(node.degree_size) AS nodes_with_degree_size
MATCH ()-[rel:SIMILAR_DISEASE]-()
RETURN
    total_nodes,
    nodes_with_degree,
    nodes_with_degree_size,
    count(rel) AS stored_similarity_relationships
"""

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(validation_query).single()

print(dict(result))

{'total_nodes': 129375, 'nodes_with_degree': 129375, 'nodes_with_degree_size': 129375, 'stored_similarity_relationships': 111402}


## 9. Close Driver

Close the Neo4j driver after analytics computation is complete.

In [9]:
driver.close()
print("Closed Neo4j driver")

Closed Neo4j driver
